In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

In [ ]:
path = "/content/drive/MyDrive/WELFake_Dataset.csv"
df = pd.read_csv(path)

In [ ]:
df.head(5)

In [ ]:
df = df.dropna(subset=['title', 'text'])
df['content'] = df['title'] + " " + df['text']
df = df[['content', 'label']].sample(15000, random_state=42)

In [ ]:
print(df["label"].value_counts())
print(df.shape)

# TOKENIZATION

In [ ]:
from transformers import DistilBertTokenizer
import torch

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_data(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

encodings = tokenize_data(df['content'])

In [ ]:
import torch
from torch.utils.data import Dataset

class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

labels = df['label'].tolist()

encodings = tokenizer(
    df['content'].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

full_dataset = FakeNewsDataset(encodings, labels)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

print(f"Dataset ready! Training samples: {train_size}, Validation samples: {val_size}")

# TRAINING AND FINE TUNING

In [ ]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
    }
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
from google.colab import files
import os

os.makedirs('fake_news_model', exist_ok=True)
model.save_pretrained('./fake_news_model')
tokenizer.save_pretrained('./fake_news_model')

!zip -r model.zip ./fake_news_model
files.download('model.zip')